In [ ]:
from thefuzz import process
import cn2an
import logging
import re
from utils import pre_process, pre_process_without_n

In [ ]:
def find_评估标准_rule_base(docs):
    logging.getLogger().setLevel(logging.ERROR)
    
    keywords = [
        "自评","巡查","监督","检查","追究","考核"
    ]
    pattern = re.compile('|'.join(keywords))
    
    keywords_fuzzy = ["责任追究","尽职免责","领导巡查","提交报告"]
    
    keywords_章节 = ["(?:^|,)\s*第.*?(章|节|点|条).*?(绩效检查|监督检查|监督管理|监督|绩效|评估|评价|考核|自评)"]
    pattern_章节 = re.compile('|'.join(keywords_章节))
    
    paragraphs = pre_process(docs)
    
    matched_paragraphs = []
    i = 0
    while i < len(paragraphs):
        paragraph = paragraphs[i]
        if pattern_章节.search(paragraph):
            temp = []
            temp += [paragraph]
            
            # number = ['1','2','3','4','5','6','7','8','9']
            number_pattern = re.compile(r'\d+')
            
            # number_chinese = ["一","二","三","四","五","六","七","八","九"]
            number_chinese_pattern = re.compile(r'(?:(?:[一二三四五六七八九]十)?[一二三四五六七八九]|十[一二三四五六七八九]?|二十|三十|四十|五十|六十|七十|八十|九十)')
            
            end_word = None
            try:
                if re.search(number_pattern, paragraph):
                    number = re.search(number_pattern, paragraph).group()
                    index = paragraph.index(number)

                    target_number = str(int(number) + 1)
                    target_number = str(target_number)

                    end_word = target_number + paragraph[index + len(number)]
                
                elif number_chinese_pattern.search(paragraph):
                    number = re.search(number_chinese_pattern, paragraph).group()
                    index = paragraph.index(number)

                    number_int = cn2an.cn2an(number)
                    number_int += 1
                    target_number = cn2an.an2cn(number_int)

                    end_word = target_number + paragraph[index + len(number)]
            except:
                pass
            
            if end_word:
                reach_end = True
                
                i += 1
                while i < len(paragraphs) and reach_end:
                    if end_word in paragraphs[i]:
                        reach_end = False
                        break
                    temp += [paragraphs[i]]
                    i += 1
                    
                if reach_end == False:
                    # add list
                    for index, sentence in enumerate(temp):
                        if index == 0:
                            matched_paragraphs.append((sentence, pattern_章节.search(paragraph).group()))
                        else:
                            matched_paragraphs.append((sentence, "章节/条款内容"))
                            
                else:
                    temp = [paragraph]
                    next_index = paragraphs.index(paragraph) + 1
                    if next_index < len(paragraphs):
                        temp += [paragraphs[next_index]]
                        for index, sentence in enumerate(temp):
                            if index == 0:
                                matched_paragraphs.append((sentence, pattern_章节.search(paragraph).group()))
                            else:
                                matched_paragraphs.append((sentence, "章节/条款内容"))
                    else:
                        matched_paragraphs.append((paragraph.strip(), pattern_章节.search(paragraph).group()))
                    i = next_index
                    
            else:
                temp = [paragraph]
                next_index = paragraphs.index(paragraph) + 1
                if next_index < len(paragraphs):
                    temp += [paragraphs[next_index]]
                    for index, sentence in enumerate(temp):
                        if index == 0:
                            matched_paragraphs.append((sentence, pattern_章节.search(paragraph).group()))
                        else:
                            matched_paragraphs.append((sentence, "章节/条款内容"))
                else:
                    matched_paragraphs.append((paragraph.strip(), pattern_章节.search(paragraph).group()))
                i = next_index
                
            matched_paragraphs[0] = (matched_paragraphs[0][0], pattern_章节.search(paragraph).group())
            
        elif pattern.search(paragraph):
            matched_paragraphs.append((paragraph.strip(), pattern.search(paragraph).group()))
        elif process.extractOne(paragraph, keywords_fuzzy)[1] >= 60:
                matched_paragraphs.append((paragraph.strip(), process.extractOne(paragraph, keywords_fuzzy)[0]))   
            
        i += 1
            
    matched_paragraphs_index = []
    for sentence in matched_paragraphs:
        begin_index = docs.find(sentence[0])
        end_index = begin_index + len(sentence[0])
        matched_paragraphs_index.append((begin_index, end_index))
        
    return matched_paragraphs, matched_paragraphs_index
